# Test the trained Persian tokenizer

Run the cells in order. This notebook loads your saved tokenizer locally; it does not download models or retrain anything.

From the repository root, launch with:

```bash
uv run --with jupyterlab jupyter lab test_tokenizer.ipynb
```

Select the Python kernel from this project's `.venv` if your editor asks for one.

In [2]:
from pathlib import Path
from html import escape
import json

import yaml
from IPython.display import HTML, display
from transformers import PreTrainedTokenizerFast

# Locate the project when launched from the repository or a subdirectory.
PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/default.yaml").is_file() and (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch Jupyter from the train-persian-tokenizer repository.")

config = yaml.safe_load((PROJECT_ROOT / "configs/default.yaml").read_text(encoding="utf-8"))
# Change this path to test a different trained tokenizer.
TOKENIZER_DIR = PROJECT_ROOT / config["project"]["output_dir"]
if not (TOKENIZER_DIR / "tokenizer.json").is_file():
    raise FileNotFoundError(f"No trained tokenizer in {TOKENIZER_DIR}. Run uv run src/train.py first.")

tokenizer = PreTrainedTokenizerFast.from_pretrained(
    str(TOKENIZER_DIR), local_files_only=True
)
print("Loaded:", TOKENIZER_DIR)
print("Vocabulary size:", len(tokenizer))
print("Maximum sequence length:", tokenizer.model_max_length)
print("Special tokens:", tokenizer.special_tokens_map)

Loaded: /home/meysam/projects/train-persian-tokenizer/artifacts/persian-bpe-32k
Vocabulary size: 32000
Maximum sequence length: 2048
Special tokens: {'bos_token': '<bos>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>'}


In [3]:
def show_table(rows):
    """Display Unicode safely without adding a pandas dependency."""
    if not rows:
        print("No rows.")
        return
    columns = list(rows[0])
    def cell(value):
        return escape(str(value))
    header = "".join(f"<th>{cell(column)}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(
            f'<td style="white-space:pre-wrap;unicode-bidi:plaintext">{cell(row[column])}</td>'
            for column in columns
        ) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


def normalized_text(text):
    # Use the normalizer saved in the tokenizer, so comparisons match its behavior.
    normalizer = tokenizer.backend_tokenizer.normalizer
    return normalizer.normalize_str(text) if normalizer is not None else text


def inspect_text(text):
    encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    ids = encoded["input_ids"]
    decoded = tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
    print("Input:     ", repr(text))
    print("Normalized:", repr(normalized_text(text)))
    print("Decoded:   ", repr(decoded))
    print("Token count:", len(ids))
    print("Exact round trip:", decoded == text)
    print("Matches normalized input:", decoded == normalized_text(text))
    show_table([
        {
            "Position": index,
            "ID": token_id,
            "Stored token": tokenizer.convert_ids_to_tokens(token_id),
            "Decoded piece": repr(tokenizer.decode([token_id], clean_up_tokenization_spaces=False)),
            "Character offsets": str((start, end)),
            "Input span": repr(text[start:end]),
        }
        for index, (token_id, (start, end)) in enumerate(zip(ids, encoded["offset_mapping"]))
    ])
    return encoded

## Try your own text

Edit `text` and rerun this cell. ZWNJ (the half-space in `می‌خواهم`) is preserved. Arabic Yeh/Kaf are normalized to Persian Yeh/Kaf, so decoded text can differ from the original while still matching the normalized input.

Byte-level BPE stores byte symbols that can look unfamiliar. A single token can contain only part of a Unicode character; decoding that token alone may show `�`. Check the full decoded sequence for correctness. Character offsets can overlap when multiple tokens represent one character.

In [ ]:
text = "من می‌خواهم یک مدل زبانی فارسی آموزش بدهم. سلام دنیا! 👋"
encoded = inspect_text(text)

## Compare example sentences

The configured evaluation sentences are included alongside normalization, whitespace, emoji, and mixed-language examples. Tokens per whitespace word is a rough size comparison; fewer tokens alone does not establish tokenizer quality.

In [ ]:
samples = list(config.get("evaluation", {}).get("test_texts", [])) + [
    "مي‌خواهم با پایتون يك مدل بسازم.",
    "می‌روم، نمی‌روم، کتاب‌ها و خانه‌های زیبا.",
    "قیمت: ۱۲۳٬۴۵۶ تومان؛ version 2.0 / Python 3.13",
    "سلام 👋 دنیا! 🌍",
    "خط اول\nخط دوم\tبا  دو فاصله",
    "",
]

results = []
for sample in samples:
    ids = tokenizer.encode(sample, add_special_tokens=False)
    decoded = tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
    word_count = len(sample.split())
    results.append({
        "Text": repr(sample),
        "Tokens": len(ids),
        "Whitespace words": word_count,
        "Tokens / word": round(len(ids) / word_count, 2) if word_count else "—",
        "Unknown tokens": ids.count(tokenizer.unk_token_id),
        "Exact round trip": decoded == sample,
        "Normalized round trip": decoded == normalized_text(sample),
    })
show_table(results)

## Special tokens and padded batches

Inspect the saved special-token IDs and compare encoding with and without automatic special tokens. Defining BOS/EOS tokens does not necessarily make the tokenizer insert them automatically.

Padding adds `<pad>` IDs to equalize lengths. The attention mask is `1` for input tokens and `0` for padding. These examples return Python lists, so PyTorch is not required.

In [ ]:
show_table([
    {"Special token": token, "ID": tokenizer.convert_tokens_to_ids(token)}
    for token in tokenizer.all_special_tokens
])
probe = "سلام دنیا!"
print("Without automatic special tokens:", tokenizer.encode(probe, add_special_tokens=False))
print("With automatic special tokens:   ", tokenizer.encode(probe, add_special_tokens=True))

batch_texts = ["سلام!", "من برنامه‌نویسی با پایتون را دوست دارم."]
if tokenizer.pad_token_id is None:
    raise ValueError("This tokenizer has no configured pad token.")
batch = tokenizer(batch_texts, padding=True, truncation=False, add_special_tokens=False)
show_table([
    {"Text": sample, "Input IDs": ids, "Attention mask": mask}
    for sample, ids, mask in zip(batch_texts, batch["input_ids"], batch["attention_mask"])
])
print("Decoded without padding:")
for decoded in tokenizer.batch_decode(
    batch["input_ids"], skip_special_tokens=True, clean_up_tokenization_spaces=False
):
    print(repr(decoded))

## Quick correctness checks

These checks exercise empty input, Persian ZWNJ, Arabic letter normalization, whitespace, numbers, mixed scripts, and emoji. Ordinary text should round-trip to the tokenizer's normalized input without unknown tokens. Literal special-token strings are excluded because they have reserved meanings.

In [ ]:
checks = [
    "", "سلام دنیا!", "می‌خواهم کتاب‌هایم را بخوانم.",
    "علي يك كتاب دارد.", "۱۲۳۴۵۶۷۸۹۰ 1234567890",
    "فارسی + English + العربية", "سلام 👋🌍", "  سلام\nدنیا\t!  ",
]
for sample in checks:
    ids = tokenizer.encode(sample, add_special_tokens=False)
    decoded = tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
    assert decoded == normalized_text(sample), (sample, decoded, normalized_text(sample))
    assert tokenizer.unk_token_id not in ids, (sample, ids)

assert len({len(ids) for ids in batch["input_ids"]}) == 1, "Padded lengths differ"
for sample, ids, mask in zip(batch_texts, batch["input_ids"], batch["attention_mask"]):
    unpadded = [token_id for token_id, keep in zip(ids, mask) if keep]
    assert unpadded == tokenizer.encode(sample, add_special_tokens=False)
    assert all(token_id == tokenizer.pad_token_id for token_id, keep in zip(ids, mask) if not keep)

print(f"Passed {len(checks)} round-trip checks and batch padding checks.")

## Saved training evaluation (optional)

This reads the evaluation created during training, if present. It does not overwrite your artifacts.

In [ ]:
report_path = TOKENIZER_DIR / "evaluation.json"
if report_path.is_file():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("Saved vocabulary size:", report.get("vocab_size"))
    show_table([
        {key: row.get(key) for key in ("text", "token_count", "round_trip_exact", "decoded")}
        for row in report.get("examples", [])
    ])
else:
    print("No saved evaluation.json; use the live results above.")